# Information Retrieval Lab: Inverted Index, BM25, and Language Model

In this lab, we build an end-to-end Information Retrieval (IR) and text classification pipeline
from scratch over GitHub Archive issues.

### Topics Covered:
1. Inverted Indexing & Postings Lists
2. Robertson-Sparck Jones IDF
3. Okapi BM25 Ranking
4. Query Likelihood Model (QLM) with Dirichlet Smoothing
5. Multinomial Naive Bayes Classifier with Laplace Smoothing

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in sys.path
sys.path.append(str(Path('.').resolve()))

import pyarrow.parquet as pq
from src.ir.index import InvertedIndex, tokenize
from src.ir.bm25 import BM25Ranker
from src.ir.lm import LanguageModelRanker
from src.ir.naive_bayes import MultinomialNaiveBayes

print('IR modules imported successfully!')

## 1. Load Curated Splits
We load the 80/10/10 train/validation/test partitions generated from the cleaned Lakehouse.

In [ ]:
splits_table = pq.read_table('data/curated/splits.parquet')
records = splits_table.to_pylist()
train_records = [r for r in records if r['split'] == 'train']
test_records = [r for r in records if r['split'] == 'test']
print(f'Total issues: {len(records)} | Train: {len(train_records)} | Test: {len(test_records)}')

## 2. Build Inverted Index
An inverted index maps terms to a postings list of `(document_id, term_frequency)`.

In [ ]:
index = InvertedIndex()
for r in train_records:
    index.add_document(r['event_id'], f"{r['title']} {r['body']}", metadata=r)
index.finalize()
print(f'Indexed documents: {index.doc_count}')
print(f'Unique vocabulary terms: {len(index.postings)}')
print(f'Average document length: {index.avg_doc_length:.2f} tokens')

## 3. Okapi BM25 Ranking
We rank documents using the Okapi BM25 scoring formula.

In [ ]:
bm25 = BM25Ranker(index, k1=1.5, b=0.75)
query = 'database connection pool timeout'
results = bm25.score_query(query, top_k=5)
for doc_id, score in results:
    meta = index.doc_metadata[doc_id]
    print(f"Score: {score:.3f} | [{meta['label']}] {meta['title']} ({meta['repo_name']})")

## 4. Query Likelihood Model (Dirichlet Prior)
We score documents using Bayesian Dirichlet smoothed unigram language models.

In [ ]:
lm = LanguageModelRanker(index, mu=2000.0)
lm_results = lm.score_query(query, top_k=5)
for doc_id, score in lm_results:
    meta = index.doc_metadata[doc_id]
    print(f"Log-Likelihood: {score:.3f} | [{meta['label']}] {meta['title']}")

## 5. Multinomial Naive Bayes Classifier
We train a generative probabilistic classifier with Laplace add-one smoothing.

In [ ]:
train_texts = [f"{r['title']} {r['body']}" for r in train_records]
train_labels = [r['label'] for r in train_records]
test_texts = [f"{r['title']} {r['body']}" for r in test_records]
test_labels = [r['label'] for r in test_records]

nb = MultinomialNaiveBayes(alpha=1.0)
nb.fit(train_texts, train_labels)
metrics = nb.evaluate(test_texts, test_labels)
print(f"Test Accuracy: {metrics['accuracy']*100:.2f}%")
print(f"Macro F1-Score: {metrics['macro_f1']*100:.2f}%")
print('\nPer-Class Metrics:')
for c, stats in metrics['per_class'].items():
    print(f"  {c:15s} P={stats['precision']:.3f} R={stats['recall']:.3f} F1={stats['f1']:.3f}")